# LDM — Results Notebook

Load a trained Latent Diffusion Model (LDMModel: frozen RRDB encoder + frozen KL-VAE +
conditional DDPM U-Net in 4x-compressed latent space) and produce:
- Single-sample comparison view
- Per-channel MAE / RMSE / PSNR
- VAE reconstruction quality
- Latent channel heatmaps (4 channels, RdBu_r)
- VAE + diffusion training curves
- Multifield physical-consistency metrics (from pre-run CSV)
- Overlay plots
- Full test-set statistics

**Before running:** fill in the paths in the next cell.
VAE must already be pretrained (`pretrain_vae()` run) before the latent diff checkpoint can be loaded.

In [ ]:
# ── USER CONFIG — edit these paths before running ─────────────────────────────

# LDM run directory (written by LDMModel training)
LDM_RUN_DIR    = "/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/ldmimplicitencoded/cs_both_n3"
# VAE pretraining run directory (written by pretrain_vae)
VAE_RUN_DIR    = "/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/vae/cs_both_n3"
# RRDB encoder run directory
ENC_RUN_DIR    = "/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/encoder/cs_both_n3"
# Data root
DATA_ROOT      = "/trace/group/forgelab/ngng/multifield/data_fields"
# Output dir from multifield_eval.py; set None to skip consistency-metric cells
EVAL_OUT_DIR   = "/trace/group/forgelab/ngng/multifield/eval_results"
EVAL_LABEL     = "LDM_both"

# ── Dataset / model config — must match training YAML ─────────────────────────
FIELD_NAMES      = ['temperature', 'liqlabel']
N_STEPS          = 3
DOWNSCALE_METHOD = 'direct'
NORMALIZE        = 'standardize'
TIMESTEPS        = 1000
SCHEDULE         = 'linear'
DEVICE           = 'cuda'

# ── Visualisation config ───────────────────────────────────────────────────────
DATA_SPLIT     = 'test'
BATCH_INDEX    = 0
SAMPLE_INDEX   = 0
CHANNEL_INDEX  = 0
SAMPLER        = 'DDIM'
DDIM_SKIP      = 50
BATCH_SIZE     = 4
T_LIQ          = 1700.0
LIQ_THR        = 0.5
EXPORT_RESULTS = False

In [ ]:
%matplotlib inline
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')  # disable CUDA on login nodes; delete this line on a GPU node

from pathlib import Path
import sys
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import torch
from torch.utils.data import DataLoader
from scipy.ndimage import gaussian_filter
from IPython.display import display as _ipy_display

# Patch plt.show() so figures render in VS Code remote / headless Jupyter
_plt_show_orig = plt.show
def _show(*args, **kwargs):
    for num in plt.get_fignums():
        _ipy_display(plt.figure(num))
    plt.close('all')
plt.show = _show

if not torch.cuda.is_available() and DEVICE == 'cuda':
    print('No GPU found — falling back to CPU (inference will be slow for large test sets)')
    DEVICE = 'cpu'

def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'setup.py').exists() and (path / 'diffusionsr').exists():
            return path
    raise RuntimeError('Could not find project root — run from within the repo')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')

## Shared Helpers

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.runners.plot_training_curves import collect_curves

def build_datasets(field_names=FIELD_NAMES, n_steps=N_STEPS):
    kw = dict(downscale_method=DOWNSCALE_METHOD, root_folder=DATA_ROOT,
              normalize=NORMALIZE, n_steps=n_steps, field_names=field_names)
    return (SimulationXZDataset(split='train', **kw),
            SimulationXZDataset(split='dev',   **kw),
            SimulationXZDataset(split='test',  **kw))

def get_batch(loader, batch_index):
    for i, batch in enumerate(loader):
        if i == batch_index: return batch
    raise IndexError(f'batch_index {batch_index} out of range')

def as_numpy(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

def display_settings(field_name):
    if field_name == 'temperature': return 'jet', 293.0, 5000.0
    elif field_name == 'liqlabel':  return 'plasma', 0.0, 1.0
    return 'viridis', None, None

def mae_rmse(pred, target):
    p, t = np.asarray(pred).ravel(), np.asarray(target).ravel()
    return {'MAE': float(np.mean(np.abs(p - t))),
            'RMSE': float(np.sqrt(np.mean((p - t)**2)))}

def psnr_val(pred, target):
    mse = np.mean((np.asarray(pred) - np.asarray(target))**2)
    if mse == 0: return float('inf')
    return float(20 * np.log10(np.max(np.abs(np.asarray(target))) / np.sqrt(mse)))

from scipy.spatial import KDTree
from scipy.ndimage import binary_erosion

def boundary_pixels(mask):
    return np.argwhere(mask & ~binary_erosion(mask, structure=np.ones((3,3))))

def chamfer_dist(mask_a, mask_b):
    b_a, b_b = boundary_pixels(mask_a), boundary_pixels(mask_b)
    if len(b_a) == 0 or len(b_b) == 0: return float('nan')
    return float((KDTree(b_b).query(b_a)[0].mean() + KDTree(b_a).query(b_b)[0].mean()) / 2)

def consistency_metrics(bin_t, bin_liq):
    inter = (bin_t & bin_liq).sum(); union = (bin_t | bin_liq).sum()
    iou = inter / union if union > 0 else float('nan')
    mse = float(np.mean((bin_t.astype(float) - bin_liq.astype(float))**2))
    return iou, mse, chamfer_dist(bin_t, bin_liq)

def overlay_plot(T_bg, T_bin, liq_bin, title, ax=None, sigma=1.5):
    standalone = ax is None
    if standalone: fig, ax = plt.subplots(figsize=(5, 4), dpi=150)
    if T_bg is not None:
        ax.imshow(T_bg.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
    ax.contour(gaussian_filter(T_bin.T.astype(float), sigma), levels=[0.5],
               colors=['red'], linewidths=[1.5], origin='lower')
    ax.contour(gaussian_filter(liq_bin.T.astype(float), sigma), levels=[0.5],
               colors=['blue'], linewidths=[1.5], origin='lower')
    ax.legend(handles=[
        plt.Line2D([0],[0], color='red',  lw=1.5, label='Binary-T (T>1700K)'),
        plt.Line2D([0],[0], color='blue', lw=1.5, label='Liqlabel (liq>0.5)'),
    ], fontsize=7, loc='upper right')
    ax.set_title(title, fontsize=8); ax.axis('off')
    if standalone: plt.tight_layout(); plt.show()

## Load LDMModel

In [ ]:
from diffusionsr.runners.train_ldm import LDMModel

train_ds, dev_ds, test_ds = build_datasets()
print(f'Train: {len(train_ds)}  Dev: {len(dev_ds)}  Test: {len(test_ds)}')
print(f'Fields: {train_ds.field_names}  |  HR shape: {train_ds.img_shape}  |  {train_ds.factor}x')

model = LDMModel(
    vae_folder=VAE_RUN_DIR,
    results_folder=LDM_RUN_DIR,
    lr_encoder_folder=ENC_RUN_DIR,
    train_dataset=train_ds,
    dev_dataset=dev_ds,
    test_dataset=test_ds,
    timesteps=TIMESTEPS,
    conditioning='implicit',
    encoding=True,
    schedule=SCHEDULE,
    device=DEVICE,
    enc_output=False,
)
model.load_saved_model()
print(f'LDMModel loaded from {LDM_RUN_DIR}')

## Select Sample and Run Inference

In [ ]:
loader = DataLoader(test_ds if DATA_SPLIT == 'test' else
                    dev_ds  if DATA_SPLIT == 'dev'  else train_ds,
                    batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
batch = get_batch(loader, BATCH_INDEX)
res, hr, true_lr, upscaled_lr = batch[:4]

hr_s  = hr[SAMPLE_INDEX:SAMPLE_INDEX+1]
lr_s  = true_lr[SAMPLE_INDEX:SAMPLE_INDEX+1]
ul_s  = upscaled_lr[SAMPLE_INDEX:SAMPLE_INDEX+1]

x_e = model.compute_x_e(lr_s, ul_s)  # latent conditioning: VAE_enc(RRDB(LR))

with torch.no_grad():
    if SAMPLER == 'DDPM':
        samples = model.batch_sample(dataset=test_ds, batch=hr_s.to(DEVICE),
                                     x_e=x_e, sampler='DDPM')
    else:
        samples = model.batch_sample(dataset=test_ds, batch=hr_s.to(DEVICE),
                                     x_e=x_e, sampler='DDIM', skip=DDIM_SKIP)

fn = test_ds.field_names
# samples contains pixel-space images (decoded by VAE inside batch_sample)
pred_phys = test_ds.unscale_data(samples[-1].cpu().numpy()[0], input_type='hr')
hr_phys   = test_ds.unscale_data(as_numpy(hr_s[0]),            input_type='hr')
lr_phys   = test_ds.unscale_data(as_numpy(lr_s[0]),            input_type='lr')
up_phys   = test_ds.unscale_data(as_numpy(ul_s[0]),            input_type='upscaled_lr')
print(f'Inference done. pred shape: {pred_phys.shape}')

## VAE Reconstruction Quality

Checks how much information is lost in the VAE encode-decode roundtrip (separate from the diffusion quality).

In [ ]:
with torch.no_grad():
    mu, logvar = model.vae.encode(hr_s.to(DEVICE).float())
    vae_recon  = model.vae.decode(mu).cpu().numpy()[0]

vae_recon_phys = test_ds.unscale_data(vae_recon, input_type='hr')

print('VAE reconstruction quality (encode → decode with mean, no sampling noise):')
import pandas as pd
rows = []
for i, fname in enumerate(fn):
    if i >= pred_phys.shape[0]: break
    m = mae_rmse(vae_recon_phys[i], hr_phys[i])
    m['PSNR'] = psnr_val(vae_recon_phys[i], hr_phys[i]); m['field'] = fname
    rows.append(m)
print(pd.DataFrame(rows)[['field','MAE','RMSE','PSNR']].to_string(index=False))

## Latent Channel Heatmaps

Visualise the 4 latent channels `z = VAE_enc(HR)` for the selected sample.

In [ ]:
with torch.no_grad():
    mu_hr, _ = model.vae.encode(hr_s.to(DEVICE).float())

z_np = as_numpy(mu_hr[0])  # (4, H/4, W/4)
n_ch = z_np.shape[0]

fig, axes = plt.subplots(1, n_ch, figsize=(4.5*n_ch, 3.5), dpi=150)
for i, ax in enumerate(axes):
    vlim = np.abs(z_np[i]).max()
    im = ax.imshow(z_np[i].T, origin='lower', cmap='RdBu_r',
                   vmin=-vlim, vmax=vlim, aspect='auto')
    ax.set_title(f'Latent ch {i}', fontsize=9); ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
fig.suptitle('VAE latent z = enc(HR) — 4 channels (H/4 × W/4)', fontsize=9)
plt.tight_layout(); plt.show()

# Also show latent conditioning z_e = enc(RRDB(LR))
with torch.no_grad():
    ze_np = as_numpy(x_e[0])
fig, axes = plt.subplots(1, n_ch, figsize=(4.5*n_ch, 3.5), dpi=150)
for i, ax in enumerate(axes):
    vlim = np.abs(ze_np[i]).max()
    im = ax.imshow(ze_np[i].T, origin='lower', cmap='RdBu_r',
                   vmin=-vlim, vmax=vlim, aspect='auto')
    ax.set_title(f'Cond z_e ch {i}', fontsize=9); ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
fig.suptitle('Latent conditioning z_e = VAE_enc(RRDB(LR))', fontsize=9)
plt.tight_layout(); plt.show()

## Comparison View

In [ ]:
ch = CHANNEL_INDEX
field = fn[ch] if ch < len(fn) else f'ch{ch}'
cmap, vmin, vmax = display_settings(field)

panels = [
    ('True LR',         lr_phys[ch]),
    ('Upscaled LR',     up_phys[ch]),
    ('Ground Truth',    hr_phys[ch]),
    ('VAE recon',       vae_recon_phys[ch]),
    ('LDM prediction',  pred_phys[ch]),
]
fig, axes = plt.subplots(1, len(panels), figsize=(4*len(panels), 4), dpi=150)
for ax, (title, data) in zip(axes, panels):
    im = ax.imshow(data.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_title(title, fontsize=9); ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
fig.suptitle(f'LDM  |  {field}  |  {DATA_SPLIT} b{BATCH_INDEX} s{SAMPLE_INDEX}', fontsize=10)
plt.tight_layout(); plt.show()

## Single-Sample Metrics

In [ ]:
rows = []
for i, fname in enumerate(fn):
    if i >= pred_phys.shape[0]: break
    m = mae_rmse(pred_phys[i], hr_phys[i]); m['PSNR'] = psnr_val(pred_phys[i], hr_phys[i]); m['field'] = fname
    rv = mae_rmse(vae_recon_phys[i], hr_phys[i]); rv['PSNR'] = psnr_val(vae_recon_phys[i], hr_phys[i]); rv['field'] = fname + ' (VAE recon)'
    bl = mae_rmse(up_phys[i], hr_phys[i]);  bl['PSNR'] = psnr_val(up_phys[i], hr_phys[i]);  bl['field'] = fname + ' (upscaled LR)'
    rows += [m, rv, bl]
print(pd.DataFrame(rows)[['field','MAE','RMSE','PSNR']].to_string(index=False))

## Multifield Consistency — Single Sample

In [ ]:
has_T   = 'temperature' in fn
has_liq = 'liqlabel'    in fn
if has_T and has_liq:
    T_pred, liq_pred = pred_phys[fn.index('temperature')], pred_phys[fn.index('liqlabel')]
    T_gt,   liq_gt   = hr_phys[fn.index('temperature')],   hr_phys[fn.index('liqlabel')]

    iou_p, mse_p, cham_p = consistency_metrics(T_pred > T_LIQ, liq_pred > LIQ_THR)
    iou_g, mse_g, cham_g = consistency_metrics(T_gt   > T_LIQ, liq_gt   > LIQ_THR)
    print(f'LDM predicted: IOU={iou_p:.4f}  MSE={mse_p:.4f}  Chamfer={cham_p:.2f} px')
    print(f'GT ref:        IOU={iou_g:.4f}  MSE={mse_g:.4f}  Chamfer={cham_g:.2f} px')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=150)
    overlay_plot(T_pred, T_pred > T_LIQ, liq_pred > LIQ_THR,
                 f'LDM (IOU={iou_p:.3f})', ax=axes[0])
    overlay_plot(T_gt,   T_gt   > T_LIQ, liq_gt   > LIQ_THR,
                 f'GT reference (IOU={iou_g:.3f})', ax=axes[1])
    plt.tight_layout(); plt.show()
else:
    print('Both temperature and liqlabel fields required.')

## Training Curves

VAE pretraining curves (`vae_train_loss.txt`, `vae_val_loss.txt`) plus LDM diffusion curves.

In [ ]:
enc_curves  = collect_curves(Path(ENC_RUN_DIR))
vae_curves  = collect_curves(Path(VAE_RUN_DIR))
ldm_curves  = collect_curves(Path(LDM_RUN_DIR))

# ── VAE curves ─────────────────────────────────────────────────────────────────
if vae_curves:
    fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
    colors = plt.cm.tab10.colors
    for i, (label, values) in enumerate(vae_curves):
        ls = '--' if ('val' in label.lower() or 'test' in label.lower()) else '-'
        ax.plot(np.arange(1, len(values)+1), values, ls=ls,
                color=colors[i % len(colors)], label=label, lw=1.6, alpha=0.9)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('VAE Pretraining Curves')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print(f'No VAE loss files found in {VAE_RUN_DIR}')

# ── Encoder + LDM curves ───────────────────────────────────────────────────────
all_curves = [('Encoder: ' + l, v) for l, v in enc_curves] + ldm_curves
if all_curves:
    fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
    colors = plt.cm.tab10.colors
    for i, (label, values) in enumerate(all_curves):
        ls = '--' if ('val' in label.lower() or 'test' in label.lower()) else '-'
        ax.plot(np.arange(1, len(values)+1), values, ls=ls,
                color=colors[i % len(colors)], label=label, lw=1.6, alpha=0.9)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('LDM Latent Diffusion Training Curves')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print(f'No LDM loss files found in {LDM_RUN_DIR}')

### W&B Training Curves (Optional Fallback)

In [ ]:
WANDB_PROJECT = 'Flow3D_SuperResolution'
WANDB_VAE_ID  = None  # W&B run ID for VAE pretraining, or None
WANDB_LDM_ID  = None  # W&B run ID for LDM training, or None

def plot_wandb_run(run_id, title):
    try:
        import wandb
        api = wandb.Api()
        run = api.run(f"{os.environ.get('WANDB_ENTITY', 'ngng-')}/{WANDB_PROJECT}/{run_id}")
        history = run.history(keys=['train_loss', 'val_loss', 'vae_train_loss', 'vae_val_loss'], pandas=True)
        fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
        for col, ls in [('train_loss','-'), ('val_loss','--'),
                        ('vae_train_loss','-'), ('vae_val_loss','--')]:
            if col in history:
                ax.plot(history[col].dropna().values, ls=ls, label=col)
        ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'W&B query failed: {e}')

if WANDB_VAE_ID: plot_wandb_run(WANDB_VAE_ID, f'W&B VAE run {WANDB_VAE_ID}')
if WANDB_LDM_ID: plot_wandb_run(WANDB_LDM_ID, f'W&B LDM run {WANDB_LDM_ID}')
if not WANDB_VAE_ID and not WANDB_LDM_ID:
    print('Set WANDB_VAE_ID / WANDB_LDM_ID to use this cell.')

## Multifield Consistency — Full Test Set (from CSV)

In [ ]:
if EVAL_OUT_DIR is None:
    print('EVAL_OUT_DIR not set — skipping.')
else:
    exp_dir  = Path(EVAL_OUT_DIR) / EVAL_LABEL
    csv_path = exp_dir / 'per_sample.csv'
    sum_path = Path(EVAL_OUT_DIR) / 'consistency_summary.csv'

    if not csv_path.exists():
        print(f'per_sample.csv not found: {csv_path}\nRun multifield_eval.py first.')
    else:
        per_sample = pd.read_csv(csv_path)
        print(f'Loaded {len(per_sample)} samples')
        print(per_sample[['iou_pred','mse_pred','cham_pred','iou_gt','mse_gt','cham_gt']].describe().round(4))

        fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=150)
        for ax, col, title in zip(axes,
                ['iou_pred','mse_pred','cham_pred'],
                ['IOU (↑)', 'MSE (↓)', 'Chamfer px (↓)']):
            valid = per_sample[col].dropna()
            ax.hist(valid, bins=30, color='teal', alpha=0.8)
            if col.replace('_pred','_gt') in per_sample.columns:
                gt_val = per_sample[col.replace('_pred','_gt')].dropna().mean()
                ax.axvline(gt_val, color='red', ls='--', lw=1.5, label=f'GT mean={gt_val:.3f}')
                ax.legend(fontsize=8)
            ax.set_title(title, fontsize=9)
        fig.suptitle(f'Per-sample distributions — {EVAL_LABEL}', fontsize=10)
        plt.tight_layout(); plt.show()

    if sum_path.exists():
        summary = pd.read_csv(sum_path)
        print('\nConsistency summary across all experiments:')
        print(summary.to_string(index=False))

## Overlay Plots

In [ ]:
if EVAL_OUT_DIR is not None:
    overlay_dir = Path(EVAL_OUT_DIR) / EVAL_LABEL / 'overlays'
    pred_pngs = sorted(overlay_dir.glob('*.png')) if overlay_dir.exists() else []
    gt_dir    = Path(EVAL_OUT_DIR) / 'gt_overlays'
    gt_pngs   = sorted(gt_dir.glob('*.png')) if gt_dir.exists() else []
    if pred_pngs:
        n = min(len(pred_pngs), 5)
        fig, axes = plt.subplots(2, n, figsize=(4.5*n, 8), dpi=120)
        for j, (pf, gf) in enumerate(zip(pred_pngs[:n],
                                         gt_pngs[:n] if gt_pngs else [None]*n)):
            axes[0,j].imshow(mpimg.imread(str(pf))); axes[0,j].axis('off')
            axes[0,j].set_title(f'LDM {pf.stem}', fontsize=7)
            if gf and Path(gf).exists():
                axes[1,j].imshow(mpimg.imread(str(gf))); axes[1,j].axis('off')
                axes[1,j].set_title(f'GT {Path(gf).stem}', fontsize=7)
            else:
                axes[1,j].axis('off')
        fig.suptitle(f'Overlay plots — {EVAL_LABEL}', fontsize=10)
        plt.tight_layout(); plt.show()
    else:
        print('No overlay PNGs found. Run multifield_eval.py --mode overlays first.')

In [ ]:
# ── Inline overlay regeneration ────────────────────────────────────────────────
INLINE_OVERLAY_INDICES = [0, 1, 2]

if has_T and has_liq:
    fig, axes = plt.subplots(len(INLINE_OVERLAY_INDICES), 2,
                             figsize=(10, 4*len(INLINE_OVERLAY_INDICES)), dpi=120, squeeze=False)
    for row, idx in enumerate(INLINE_OVERLAY_INDICES):
        b = test_ds[idx]
        _, h, tl, ul = [torch.tensor(x).unsqueeze(0) for x in b[:4]]
        xe = model.compute_x_e(tl, ul)
        with torch.no_grad():
            samp = model.batch_sample(dataset=test_ds, batch=h.to(DEVICE),
                                      x_e=xe, sampler='DDIM', skip=DDIM_SKIP)
        p = test_ds.unscale_data(samp[-1].cpu().numpy()[0], input_type='hr')
        g = test_ds.unscale_data(as_numpy(h[0]),            input_type='hr')
        T_p, liq_p = p[fn.index('temperature')], p[fn.index('liqlabel')]
        T_g, liq_g = g[fn.index('temperature')], g[fn.index('liqlabel')]
        iou, _, _ = consistency_metrics(T_p > T_LIQ, liq_p > LIQ_THR)
        overlay_plot(T_p, T_p > T_LIQ, liq_p > LIQ_THR,
                     f'LDM s{idx} (IOU={iou:.3f})', ax=axes[row,0])
        overlay_plot(T_g, T_g > T_LIQ, liq_g > LIQ_THR,
                     f'GT s{idx}', ax=axes[row,1])
    plt.tight_layout(); plt.show()
else:
    print('Overlay plots require both temperature and liqlabel fields.')

## Full Test-Set Statistics

In [ ]:
from diffusionsr.analysis.analysis_functions import get_profile

ANALYSIS_MAX_BATCH = None
ANALYSIS_CH        = 0
MELT_THRESHOLD     = 1900.0

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
maes, rmses, profile_maes = [], [], []
iou_list, mse_list, cham_list = [], [], []
iou_gt_list, mse_gt_list, cham_gt_list = [], [], []
vae_maes = []

for i, batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i >= ANALYSIS_MAX_BATCH: break
    res_b, hr_b, lr_b, ul_b = batch[:4]
    xe = model.compute_x_e(lr_b, ul_b)
    with torch.no_grad():
        samps = model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                   x_e=xe, sampler='DDIM', skip=DDIM_SKIP)
        mu_hr, _ = model.vae.encode(hr_b.to(DEVICE).float())
        vae_r    = model.vae.decode(mu_hr).cpu().numpy()
    for s in range(hr_b.shape[0]):
        p = test_ds.unscale_data(samps[-1].cpu().numpy()[s], input_type='hr')
        g = test_ds.unscale_data(as_numpy(hr_b[s]),          input_type='hr')
        m = mae_rmse(p[ANALYSIS_CH], g[ANALYSIS_CH])
        maes.append(m['MAE']); rmses.append(m['RMSE'])
        vae_maes.append(mae_rmse(test_ds.unscale_data(vae_r[s], input_type='hr')[ANALYSIS_CH],
                                 g[ANALYSIS_CH])['MAE'])
        try:
            pp = get_profile(p[ANALYSIS_CH], threshold=MELT_THRESHOLD)
            gp = get_profile(g[ANALYSIS_CH], threshold=MELT_THRESHOLD)
            if pp and gp: profile_maes.append(float(np.mean(np.abs(np.array(pp) - np.array(gp)))))
        except Exception: pass
        if has_T and has_liq:
            T_p, liq_p = p[fn.index('temperature')], p[fn.index('liqlabel')]
            T_g_c, liq_g_c = g[fn.index('temperature')], g[fn.index('liqlabel')]
            iou, mse_c, cham = consistency_metrics(T_p > T_LIQ, liq_p > LIQ_THR)
            iou_gt, mse_gt_c, cham_gt = consistency_metrics(T_g_c > T_LIQ, liq_g_c > LIQ_THR)
            iou_list.append(iou); mse_list.append(mse_c); cham_list.append(cham)
            iou_gt_list.append(iou_gt); mse_gt_list.append(mse_gt_c); cham_gt_list.append(cham_gt)

print(f'Test-set results (n={len(maes)}):')
print(f'  MAE  = {np.nanmean(maes):.4f} ± {np.nanstd(maes):.4f}')
print(f'  RMSE = {np.nanmean(rmses):.4f} ± {np.nanstd(rmses):.4f}')
print(f'  VAE recon MAE = {np.nanmean(vae_maes):.4f} ± {np.nanstd(vae_maes):.4f}  (encode-decode ceiling)')
if profile_maes:
    print(f'  Profile MAE = {np.nanmean(profile_maes):.4f} ± {np.nanstd(profile_maes):.4f}')
if iou_list:
    print(f'  IOU pred     = {np.nanmean(iou_list):.4f} ± {np.nanstd(iou_list):.4f}')
    print(f'  IOU GT       = {np.nanmean(iou_gt_list):.4f} ± {np.nanstd(iou_gt_list):.4f}  (GT self-consistency ceiling)')
    print(f'  MSE pred     = {np.nanmean(mse_list):.4f} ± {np.nanstd(mse_list):.4f}')
    print(f'  MSE GT       = {np.nanmean(mse_gt_list):.4f} ± {np.nanstd(mse_gt_list):.4f}')
    print(f'  Chamfer pred = {np.nanmean(cham_list):.2f} ± {np.nanstd(cham_list):.2f} px')
    print(f'  Chamfer GT   = {np.nanmean(cham_gt_list):.2f} ± {np.nanstd(cham_gt_list):.2f} px')

## Export

In [ ]:
if EXPORT_RESULTS:
    out = Path(LDM_RUN_DIR) / f'results_{DATA_SPLIT}_b{BATCH_INDEX}_s{SAMPLE_INDEX}.npz'
    np.savez(str(out), true_lr=lr_phys, upscaled_lr=up_phys,
             ground_truth=hr_phys, vae_reconstruction=vae_recon_phys,
             prediction=pred_phys)
    print(f'Saved → {out}')
else:
    print('Set EXPORT_RESULTS = True to save.')